# Event Calibration and Fitting Comparison

This notebook builds a PeakLoc event-background calibration from the paired **dark** and **laser-on blank** recordings in `data/Calibration`. It then reads one second from an experimental RAW recording, detects one shared set of candidate emitters, and fits every ROI both with and without the calibration.

Using identical events and ROIs in both fits is important: differences in position, uncertainty, likelihood, or acceptance can then be attributed to the calibration model rather than to a different set of detected peaks.

## 1. Setup and reusable helpers

Run this notebook with the repository's pixi kernel. RAW files are streamed in time windows, so neither the calibration recordings nor the five-minute experiment need to be loaded in full. PeakLoc uses `image[y, x]`; plot overlays therefore use `scatter(x, y)`.

In [ ]:
from __future__ import annotations

from dataclasses import replace
import gc
import json
import os
from pathlib import Path
import site
import sys

# Prevent user-site packages from shadowing the versions locked by pixi.
os.environ["PYTHONNOUSERSITE"] = "1"
USER_SITE = site.getusersitepackages()
if USER_SITE in sys.path:
    sys.path.remove(USER_SITE)

from IPython.display import Markdown, display  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402


def find_repo_root(start: Path | None = None) -> Path:
    path = (Path.cwd() if start is None else start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pixi.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing pixi.toml")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

from calibration_scripts.build_event_calibration import (  # noqa: E402
    build_rate_maps,
    write_event_calibration,
)
from localization_scripts.calibration import (  # noqa: E402
    NullCalibration,
    load_calibration,
)
from localization_scripts.event_array_processing import (  # noqa: E402
    array_to_polarity_map,
    array_to_time_map,
    create_convolved_signals,
)
from localization_scripts.localization_fitting import (  # noqa: E402
    LocalizationTables,
    localization_uncertainty_px,
    localize_rois_with_attempts,
)
from localization_scripts.peak_finding import (  # noqa: E402
    create_peak_lists,
    find_local_max_peak,
    find_peaks_parallel,
    group_timestamps_by_coordinate,
)
from localization_scripts.pipeline_config import PeakLocConfig  # noqa: E402
from localization_scripts.roi_generation import generate_rois  # noqa: E402
from scripts.raw_to_video import open_raw_reader  # noqa: E402

EVENT_DTYPE = np.dtype([("x", "u2"), ("y", "u2"), ("p", "i1"), ("t", "u8")])


def only_raw_file(folder: Path) -> Path:
    paths = sorted(folder.glob("*.raw"))
    if len(paths) != 1:
        raise ValueError(f"Expected one RAW file in {folder}, found {len(paths)}")
    return paths[0]


def read_raw_window(
    path: Path,
    *,
    start_us: int,
    duration_us: int,
    read_step_us: int = 1_000_000,
) -> tuple[np.ndarray, tuple[int, int]]:
    if start_us < 0 or duration_us <= 0 or read_step_us <= 0:
        raise ValueError("start_us must be non-negative and durations must be positive")

    reader = open_raw_reader(path, max_events_buffer=100_000_000)
    sensor_shape = reader.get_size()
    remaining_skip = start_us
    while remaining_skip > 0 and not reader.is_done():
        step = min(read_step_us, remaining_skip)
        reader.load_delta_t(step)
        remaining_skip -= step

    chunks: list[np.ndarray] = []
    remaining_read = duration_us
    while remaining_read > 0 and not reader.is_done():
        step = min(read_step_us, remaining_read)
        events = reader.load_delta_t(step)
        if events.size:
            chunks.append(events.astype(EVENT_DTYPE, copy=True))
        remaining_read -= step

    events = np.concatenate(chunks) if chunks else np.empty(0, dtype=EVENT_DTYPE)
    return events, sensor_shape


def detect_rois(events: np.ndarray, config: PeakLocConfig) -> np.ndarray:
    if events.size == 0:
        raise ValueError("The selected recording window contains no events")

    coordinates = np.unique(
        np.column_stack((events["y"], events["x"])).astype(np.int32), axis=0
    )
    polarity_map, max_events_per_pixel = array_to_polarity_map(events, coordinates)
    time_polarity_map = array_to_time_map(events)

    convolved_length = int(
        max_events_per_pixel * 2 * (config.convolution_roi_radius * 2 + 1) ** 2
    )
    times, cumulative_signals, convolved_coordinates = create_convolved_signals(
        polarity_map, coordinates, convolved_length, config.num_cores
    )
    peak_candidates = find_peaks_parallel(
        times,
        cumulative_signals,
        convolved_coordinates,
        config.num_cores,
        prominence=config.prominence,
        interpolation_coefficient=config.interpolation_coefficient,
        cutoff_event_count=config.peak_min_event_count,
        spline_smooth=config.spline_smooth,
    )
    peaks, prominences, on_times, peak_coordinates = create_peak_lists(peak_candidates)
    grouped_peaks = group_timestamps_by_coordinate(
        peak_coordinates, peaks, prominences, on_times
    )
    unique_peaks = find_local_max_peak(
        grouped_peaks,
        threshold=config.peak_time_threshold,
        neighbors=config.peak_neighbors,
    )
    rois = generate_rois(
        unique_peaks,
        time_polarity_map,
        roi_rad=config.roi_radius,
        min_x=0,
        min_y=0,
        num_cores=config.num_cores,
        max_x=config.sensor_width - 1,
        max_y=config.sensor_height - 1,
        polarity_time_gate_us=config.polarity_time_gate_us,
    )
    del polarity_map, time_polarity_map, times, cumulative_signals
    gc.collect()
    return rois


print(f"Repository: {REPO_ROOT}")

## 2. Build the calibration file

The dark recording estimates spontaneous sensor activity. The laser-on blank estimates background under illumination without fluorophores. PeakLoc stores positive and negative event-rate maps separately, together with hot-pixel and valid-pixel masks.

The default uses the first 10 seconds of each recording. Increase `CALIBRATION_DURATION_US` for a lower-noise estimate at the cost of runtime and memory. A pixel is marked hot when its total rate lies above the 99.9th percentile in either recording.

In [ ]:
CALIBRATION_ROOT = REPO_ROOT / "data" / "Calibration"
CALIBRATION_DURATION_US = 10_000_000
CALIBRATION_PATH = CALIBRATION_ROOT / "event_calibration.npz"
HOT_PIXEL_QUANTILE = 0.999

dark_path = only_raw_file(CALIBRATION_ROOT / "dark")
blank_path = only_raw_file(CALIBRATION_ROOT / "blank")
dark_events, dark_shape = read_raw_window(
    dark_path, start_us=0, duration_us=CALIBRATION_DURATION_US
)
blank_events, blank_shape = read_raw_window(
    blank_path, start_us=0, duration_us=CALIBRATION_DURATION_US
)
if dark_shape != blank_shape:
    raise ValueError(
        f"Calibration sensor shapes differ: {dark_shape} and {blank_shape}"
    )

dark_maps = build_rate_maps(
    dark_events, dark_shape, hot_pixel_quantile=HOT_PIXEL_QUANTILE
)
blank_maps = build_rate_maps(
    blank_events, blank_shape, hot_pixel_quantile=HOT_PIXEL_QUANTILE
)
write_event_calibration(
    CALIBRATION_PATH,
    dark_maps=dark_maps,
    blank_maps=blank_maps,
    pixel_size_nm=67.0,
    sensor_model="Prophesee event camera",
    calibration_id=f"dark-blank-{CALIBRATION_DURATION_US / 1e6:g}s",
)
hot_pixel_count = int(
    np.count_nonzero(dark_maps.hot_pixel_mask | blank_maps.hot_pixel_mask)
)
print(f"Dark events: {dark_events.size:,}")
print(f"Laser-on blank events: {blank_events.size:,}")
print(f"Sensor shape: {dark_shape} as image[y, x]")
print(f"Hot pixels: {hot_pixel_count:,} ({hot_pixel_count / np.prod(dark_shape):.3%})")
print(f"Wrote calibration: {CALIBRATION_PATH.relative_to(REPO_ROOT)}")

del dark_events, blank_events
gc.collect()

In [ ]:
# 3. Fit the same one-second recording chunk with and without calibration.
RECORDING_FOLDER = (
    REPO_ROOT / "data" / "5_minute_Recording_Rapid_Switching_From_Power_Increase"
)
RECORDING_START_US = 30_000_000
RECORDING_DURATION_US = 1_000_000
recording_path = only_raw_file(RECORDING_FOLDER)
events, sensor_shape = read_raw_window(
    recording_path,
    start_us=RECORDING_START_US,
    duration_us=RECORDING_DURATION_US,
)

config_payload = json.loads((REPO_ROOT / "config.json").read_text(encoding="utf-8"))
config_payload.update(
    {
        "input_folder": str(RECORDING_FOLDER),
        "slice_start": RECORDING_START_US,
        "slice_duration": RECORDING_DURATION_US,
        # One core avoids worker environments shadowed by user-site packages.
        "num_cores": 1,
        "sensor_height": sensor_shape[0],
        "sensor_width": sensor_shape[1],
        "plot_result": False,
        "cleanup_temp_outputs": False,
        "qc_enabled": False,
        "allow_uncalibrated": False,
        "calibration_path": str(CALIBRATION_PATH),
    }
)
calibrated_config = PeakLocConfig.from_mapping(config_payload)
uncalibrated_config = replace(
    calibrated_config, calibration_path=None, allow_uncalibrated=True
)
calibration = load_calibration(CALIBRATION_PATH, sensor_shape, allow_uncalibrated=False)
uncalibrated_model = NullCalibration(sensor_shape)

print(f"Loaded {events.size:,} events; detecting one shared ROI set ...")
rois = detect_rois(events, calibrated_config)
print(f"Detected {rois.size:,} ROIs; fitting both background models ...")
uncalibrated = localize_rois_with_attempts(
    rois, uncalibrated_config, uncalibrated_model
)
calibrated = localize_rois_with_attempts(rois, calibrated_config, calibration)


def median_or_nan(values: np.ndarray) -> float:
    finite = np.asarray(values, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    return float(np.median(finite)) if finite.size else float("nan")


def fit_summary(label: str, tables: LocalizationTables) -> list[str]:
    attempted = tables.attempted
    accepted = tables.filtered
    uncertainty_nm = (
        localization_uncertainty_px(accepted) * calibrated_config.optical_pixel_size_nm
        if accepted.size
        else np.array([])
    )
    success_fraction = (
        float(np.mean(attempted["fit_success"])) if attempted.size else float("nan")
    )
    nll = attempted["nll_per_event"] if attempted.size else np.array([])
    return [
        label,
        f"{attempted.size:,}",
        f"{accepted.size:,}",
        f"{success_fraction:.1%}",
        f"{median_or_nan(uncertainty_nm):.2f}",
        f"{median_or_nan(nll):.3f}",
    ]


summary_rows = [
    fit_summary("Uncalibrated", uncalibrated),
    fit_summary("Calibrated", calibrated),
]
summary_markdown = [
    "| Model | Attempted | Accepted | Fit success | Median uncertainty (nm) | Median NLL/event |",
    "|---|---:|---:|---:|---:|---:|",
]
summary_markdown.extend("| " + " | ".join(row) + " |" for row in summary_rows)
display(Markdown("\n".join(summary_markdown)))

event_frame = np.zeros(sensor_shape, dtype=np.uint32)
np.add.at(event_frame, (events["y"], events["x"]), 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, title, localizations, color in [
    (axes[0, 0], "Uncalibrated accepted fits", uncalibrated.filtered, "cyan"),
    (axes[0, 1], "Calibrated accepted fits", calibrated.filtered, "lime"),
]:
    ax.imshow(np.log1p(event_frame), cmap="gray", origin="upper")
    if localizations.size:
        ax.scatter(
            localizations["x"],
            localizations["y"],
            s=10,
            facecolors="none",
            edgecolors=color,
            linewidths=0.7,
        )
    ax.set(title=title, xlabel="x (pixel)", ylabel="y (pixel)")

pair_count = min(uncalibrated.attempted.size, calibrated.attempted.size)
if pair_count:
    delta_x = (
        calibrated.attempted["x"][:pair_count]
        - uncalibrated.attempted["x"][:pair_count]
    )
    delta_y = (
        calibrated.attempted["y"][:pair_count]
        - uncalibrated.attempted["y"][:pair_count]
    )
    displacement_nm = (
        np.hypot(delta_x, delta_y) * calibrated_config.optical_pixel_size_nm
    )
    displacement_nm = displacement_nm[np.isfinite(displacement_nm)]
    axes[1, 0].hist(displacement_nm, bins=40, color="#3b82f6", alpha=0.85)
    axes[1, 0].axvline(
        median_or_nan(displacement_nm), color="black", linestyle="--", label="median"
    )
    axes[1, 0].legend()
axes[1, 0].set(
    title="Position change for matched ROI fits",
    xlabel="Calibrated vs uncalibrated displacement (nm)",
    ylabel="ROI count",
)

for label, localizations, color in [
    ("Uncalibrated", uncalibrated.filtered, "#f97316"),
    ("Calibrated", calibrated.filtered, "#16a34a"),
]:
    if localizations.size:
        uncertainty_nm = (
            localization_uncertainty_px(localizations)
            * calibrated_config.optical_pixel_size_nm
        )
        uncertainty_nm = uncertainty_nm[np.isfinite(uncertainty_nm)]
        axes[1, 1].hist(
            uncertainty_nm,
            bins=40,
            histtype="step",
            linewidth=1.8,
            label=label,
            color=color,
        )
axes[1, 1].set(
    title="Accepted-fit localization uncertainty",
    xlabel="Worst-axis 1-sigma uncertainty (nm)",
    ylabel="Localization count",
)
axes[1, 1].legend()
plt.show()

## Reading the comparison

- **Accepted** counts can change because calibrated fits mask hot pixels and use spatially varying expected background rates.
- **NLL/event** compares model agreement on a per-event scale; lower values indicate a better likelihood fit, but interpret them together with fit success and uncertainty.
- **Matched displacement** measures how much the estimated emitter position changes when only the background model changes.
- **Uncertainty** is the worst-axis one-sigma estimate converted with the configured 67 nm pixel size.

The second beginning at 30 seconds is selected by default to avoid the large illumination and focus-flash transients near the start. Change `RECORDING_START_US` and rerun the comparison cell to inspect another interval without rebuilding the calibration.